# 🟡 Lesson 14 — Geemap

**Level: Intermediate** · Interactive maps for Earth Engine in Jupyter — inspect, compare and export imagery visually.

*Part of the companion package for [python_for_geologists](https://github.com/kevinalexandr19/python_for_geologists) by Kevin Alexander Gomez.*

> ⚠️ **This notebook is not executed in this repo** — it needs Earth Engine authentication (lesson 13) plus `pip install geemap`. The code is complete and ready to run once you have set it up.

In [ ]:
import ee
import geemap

ee.Initialize(project="your-project-id")

## 1. An interactive map in two lines

In [ ]:
m = geemap.Map(center=[-16.3, -71.4], zoom=9)   # Arequipa region
m

## 2. Add Earth Engine layers with styling

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003")
vis = {"min": 0, "max": 5500,
       "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"]}
m.add_layer(srtm, vis, "SRTM elevation")
m.add_colorbar(vis, label="elevation (m)")
m

## 3. Split-map comparison — DEM vs satellite

In [ ]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate("2024-06-01", "2024-09-01")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
        .median())

left  = geemap.ee_tile_layer(srtm, vis, "DEM")
right = geemap.ee_tile_layer(s2, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Sentinel-2")
m2 = geemap.Map(center=[-16.3, -71.4], zoom=10)
m2.split_map(left, right)
m2

## 4. Draw a region on the map, then use it in code

In [ ]:
# draw a rectangle with the toolbar first, then:
roi = m.user_roi                        # the drawn geometry as ee.Geometry
stats = srtm.reduceRegion(ee.Reducer.mean(), roi, scale=90)
print("mean elevation of drawn ROI:", stats.getInfo())

## 5. Download imagery for offline work

In [ ]:
geemap.ee_export_image(srtm.clip(roi), filename="srtm_roi.tif", scale=30, region=roi)
# -> then open it with rasterio / rioxarray (lessons 09-10)!

### ✏️ Try it
1. Add the `JAXA/ALOS/AW3D30/V3_2` DEM as a second layer and toggle between them.
2. Use `m.add_basemap('HYBRID')` and digitise the outline of a volcanic edifice.

📚 Docs: https://geemap.org/